In [1]:
# --- AGGIORNAMENTO: Import e Costanti ---
from IPython.display import display, clear_output
import numpy as np
import ipywidgets as widgets
import pythreejs as p3j
import tempNcolor as tc
import number_formatting as nf
import starlib as star 

L_SUN = star.L_Sun 
T_SUN = star.Te_Sun 
M_SUN = 4.83  # Magnitudine assoluta del Sole

# --- FUNZIONI DI CALCOLO POTENZIATE ---
def get_spectral_class(temp_kelvin):
    if temp_kelvin > 30000: return "O (Blu - Caldissima)"
    if temp_kelvin > 10000: return "B (Azzurra)"
    if temp_kelvin > 7500:  return "A (Bianca)"
    if temp_kelvin > 6000:  return "F (Bianco-Gialla)"
    if temp_kelvin > 5200:  return "G (Gialla - Tipo Sole)"
    if temp_kelvin > 3700:  return "K (Arancione)"
    return "M (Rossa - Fredda)"

def calculate_m_abs(l_ratio):
    if l_ratio <= 0: return 20.0
    return M_SUN - 2.5 * np.log10(l_ratio)

# --- LOGICA DI AGGIORNAMENTO UNIFICATA ---
def Refresh(change=None):
    # 1. Calcoli Fisici
    t_ratio = Temp.value
    r_ratio = Rad.value
    l_ratio = (r_ratio**2.0) * (t_ratio**4.0)
    t_kelvin = int(t_ratio * T_SUN)
    m_abs = calculate_m_abs(l_ratio)
    
    # 2. Aggiornamento Widget Testuali
    t_star_report.value = f"{t_kelvin} K"
    spec_class_report.value = get_spectral_class(t_kelvin)
    L_Ratio_report.value = f"{l_ratio:.2f}"
    M_Abs_report.value = f"{m_abs:.2f}"
    
    # 3. Aggiornamento 3D (pythreejs)
    hex_color = tc.rgb2hex(tc.temp2rgb(t_kelvin))[0]
    star_sphere.material.color = hex_color
    star_sphere.scale = (r_ratio, r_ratio, r_ratio)

# --- DEFINIZIONE WIDGET ---
Rad = widgets.FloatSlider(description='Raggio (R/R☉)', min=0.1, max=20.0, step=0.1, value=1.0)
Temp = widgets.FloatSlider(description='Temp (T/T☉)', min=0.5, max=7.0, step=0.1, value=1.0)

t_star_report = widgets.Text(description='Temp (K):', disabled=True)
spec_class_report = widgets.Text(description='Classe:', disabled=True)
L_Ratio_report = widgets.Text(description='Lum (L/L☉):', disabled=True)
M_Abs_report = widgets.Text(description='Mag Ass (M):', disabled=True)

# --- SETUP SCENA 3D ---
star_geom = p3j.SphereBufferGeometry(radius=1, widthSegments=32, heightSegments=16)
star_mat = p3j.MeshBasicMaterial(color='#fff4ea')
star_sphere = p3j.Mesh(geometry=star_geom, material=star_mat, position=[0, 0, 0])

# Sole di riferimento (fisso e semitrasparente)
sun_mat = p3j.MeshBasicMaterial(color='#fff4ea', opacity=0.3, transparent=True)
sun_sphere = p3j.Mesh(geometry=star_geom, material=sun_mat, position=[25, 0, 0])

scene = p3j.Scene(children=[star_sphere, sun_sphere, p3j.AmbientLight(color='white')], background='black')
camera = p3j.PerspectiveCamera(position=[50, 20, 50], up=[0, 1, 0])
renderer = p3j.Renderer(camera=camera, scene=scene, controls=[p3j.OrbitControls(controlling=camera)], width=400, height=400)

# Collegamento osservatori
Rad.observe(Refresh, names='value')
Temp.observe(Refresh, names='value')

# --- LAYOUT FINALE ---
controlli = widgets.VBox([Rad, Temp])
report = widgets.VBox([t_star_report, spec_class_report, L_Ratio_report, M_Abs_report])
interfaccia = widgets.VBox([
    widgets.HTML("<h2>🔬 Laboratorio: Identikit Fisico della Stella</h2>"),
    renderer,
    widgets.HBox([controlli, report])
])

display(interfaccia)
Refresh() # Inizializzazione

# Attività 1: La Stella "Invisibile" (Proxima Centauri)

## Dati: Proxima Centauri ha un raggio di 0.15 $R_\odot$ e una temperatura di circa 3000 K (circa 0.5 $T_\odot$).
## Sfida: Imposta questi valori nel simulatore.Domanda: Qual è la sua Magnitudine Assoluta? 
### Confrontala con quella del Sole ($M = 4.83$). Riusciresti a vederla a occhio nudo se fosse alla stessa distanza del Sole?

### Concetto: Anche se è la stella più vicina, la sua $M$ è altissima (molto debole intrinsecamente).

# Attività 2: Diventare una Supergigante (Rigel)
## Dati: Rigel è una supergigante blu con $M \approx -6.7$.
## Sfida: Sapendo che Rigel è molto calda (imposta Temperatura a 2.1), muovi il cursore del Raggio finché la Magnitudine Assoluta nel simulatore non arriva a -6.7.

### Domanda: Quanto deve essere grande il raggio?

### Riflessione: Una differenza di magnitudine da 5 a -6 sembra piccola numericamente, ma guarda quanto cambia la Luminosità ($L$)!

# Attività 3: La "Regola del 2.5"
## Obiettivo: Capire la scala logaritmica.
### Esperimento: 1. Imposta la stella come il Sole ($R=1, T=1$). Segna $M$.2. Aumenta il raggio finché la Luminosità non diventa 100 volte quella del Sole ($L=100$). 
### Osservazione: Di quanto è scesa la Magnitudine Assoluta? (Dovrebbe essere scesa di esattamente 5 magnitudini).Formula da spiegare: Ogni 5 magnitudini di differenza, la luminosità cambia di 100 volte.